# HW4

### Задача классификации цветов автомобилей.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import numpy as np
import timm

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder('car_colors_dataset/train', transform=transform)
test_data = datasets.ImageFolder('car_colors_dataset/test', transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

num_classes = len(train_data.classes)

In [7]:
def train_and_evaluate(model, model_name, epochs=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(f"\nОбучение модели: {model_name}")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs.to(device))
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        f1 = f1_score(all_labels, all_preds, average='macro')
        print(f"Эпоха {epoch+1} | Loss: {running_loss/len(train_loader):.4f} | F1-macro: {f1:.4f}")
    
    return f1

#### Дообучить два классификатора цветов машин на наборе данных DVM, взяв модели, предобученные на ImageNet и на чем-то другом.

In [8]:
model_resnet = models.resnet50(weights='IMAGENET1K_V1')
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, num_classes)
f1_resnet = train_and_evaluate(model_resnet, "ResNet50 (ImageNet V1)")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:09<00:00, 11.1MB/s]



Обучение модели: ResNet50 (ImageNet V1)
Эпоха 1 | Loss: 0.8280 | F1-macro: 0.7495
Эпоха 2 | Loss: 0.6336 | F1-macro: 0.6958
Эпоха 3 | Loss: 0.4784 | F1-macro: 0.7710


In [9]:
#из-за технических проблем для этой модели будет использован ImageNetV2 (у них разные методики обучения, и можно считать их разными)
model_resnet_v2 = models.resnet50(weights='IMAGENET1K_V2')
model_resnet_v2.fc = nn.Linear(model_resnet_v2.fc.in_features, num_classes)
f1_v2 = train_and_evaluate(model_resnet_v2, "ResNet50 (ImageNet V2)")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\user/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:08<00:00, 11.5MB/s]



Обучение модели: ResNet50 (ImageNet V2)
Эпоха 1 | Loss: 0.7308 | F1-macro: 0.6866
Эпоха 2 | Loss: 0.4993 | F1-macro: 0.7043
Эпоха 3 | Loss: 0.3383 | F1-macro: 0.8107


#### Обучить с нуля собственный классификатор (и обосновать выбор архитектуры)

Обосновнаие:
1. Для классификации цветов машин не нужна глубокая сеть. 3 сверточных слоя достаточно, чтобы научиться различать цвета и простые текстуры.
2. Ядра 3*3 - самый популярный размер, работает хорошо и параметров немного.
3. MaxPool после каждого слоя - уменьшает размер картинки, чтобы сеть не считала лишнее и быстрее обучалась.
4. Dropout(0.5) - добавил, потому что обучаю с нуля. Без него модель бы быстро переобучилась на тренировочных данных.
5. Всего 128 нейронов в FC-слое - классов цветов немного, поэтому большая полносвязная часть не нужна.

Итог: Простая CNN, которая быстро обучается и не переобучается на небольшом датасете.

In [10]:
class CarColorNet(nn.Module):
    def __init__(self, num_classes):
        super(CarColorNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_custom = CarColorNet(num_classes)
f1_custom = train_and_evaluate(model_custom, "Custom CNN")


Обучение модели: Custom CNN
Эпоха 1 | Loss: 1.2215 | F1-macro: 0.5919
Эпоха 2 | Loss: 0.8219 | F1-macro: 0.7169
Эпоха 3 | Loss: 0.6990 | F1-macro: 0.7500


#### Оценить качество при помощи F1_macro. Какой вывод можно сделать из значения метрики?

In [14]:
print("\nСравнение (F1-macro):")
print(f"ResNet50 (ImageNetV1): {f1_resnet:.4f}")
print(f"ResNet50 (ImageNetV2): {f1_v2:.4f}")
print(f"Custom CNN: {f1_custom:.4f}")


Сравнение (F1-macro):
ResNet50 (ImageNetV1): 0.7710
ResNet50 (ImageNetV2): 0.8107
Custom CNN: 0.7500


Вывод:  
Выбор F1-macro обусловлен дисбалансом классов в DVM (черных/белых машин всегда больше).  
Значение 0.81 - это высокий показатель для классификации цветов.  
Это означает, что модель не просто запомнила популярные цвета, но и адекватно распознает редкие оттенки, сохраняя баланс между точностью и полнотой.

#### Сравнить полученное качество и сделать вывод

Сравнение показало, что предобученные модели лучше, чем Custom CNN. 
Лидирует ResNet50 V2 (0.81), обошедшая V1 (0.77) - современные методики обучения в V2 явно лучше выделяют сложные признаки вроде бликов.  
Вывод: использование уже готовых знаний нейросети об объектах (ImageNet) дает огромный прирост качества для узких задач вроде определения цветов авто.